In [1]:
import pandas as pd
import numpy as np
import itertools
from sklearn.preprocessing import MinMaxScaler

In [2]:
data_path = r"/Users/ysl/Documents/DefiTradingBot/PairsTrading/binance_features_all_1d_interval.feather"
df = pd.read_feather(data_path)
columns = ["DateTime", "StockID", "Open", "High", "Low", "Close", "Vol", "Ret"]

full_df = df.sort_values(["DateTime", "StockID"]).reset_index(drop=True)
full_df = full_df.rename(columns={"StockID": "AssetName"})
unique_assets = full_df["AssetName"].unique()
# filter on USDT 
usdt_df = full_df.loc[full_df["AssetName"].str.contains("USDT")]


In [3]:
print(f"Unique Assets: {len(unique_assets)}")
print(f"Assets with USDT: {len(usdt_df["AssetName"].unique())}")

Unique Assets: 1454
Assets with USDT: 412


In [4]:
def zscore(series, window=20):
    return (series - series.rolling(window).mean()) / series.rolling(window).std()

def sharpe_ratio(returns, risk_free_rate=0.0422):
    # Assumes daily returns, annualizes by sqrt(252)
    excess_ret = returns - risk_free_rate/252
    return np.mean(excess_ret) / np.std(excess_ret) * np.sqrt(252) if np.std(excess_ret) > 0 else np.nan

In [28]:
# HYPER parameters
TRAIN_DATA = 0.6
VALIDATION_DATA = 0.3
TEST_DATA = 0.1
# 
train_cutoff_date = usdt_df["DateTime"].iloc[int(len(usdt_df) * TRAIN_DATA)]
validation_cutoff_date = usdt_df["DateTime"].iloc[int(len(usdt_df) * (TRAIN_DATA + VALIDATION_DATA))]
train_df_idx = usdt_df["DateTime"] < train_cutoff_date
validation_df_idx = (usdt_df["DateTime"] >= train_cutoff_date) & (usdt_df["DateTime"] < validation_cutoff_date)
test_df_idx = usdt_df["DateTime"] >= validation_cutoff_date
train_df = usdt_df.loc[train_df_idx]
validation_df = usdt_df.loc[validation_df_idx]
test_df = usdt_df.loc[test_df_idx]

In [29]:
# 1. scale the data
scaler = MinMaxScaler()
scaler.fit(train_df[["Close"]])
train_df.loc[:, "Close"] = scaler.transform(train_df[["Close"]])
validation_df.loc[:, "Close"] = scaler.transform(validation_df[["Close"]])
test_df.loc[:, "Close"] = scaler.transform(test_df[["Close"]])

In [30]:
pivot_df = train_df.pivot(index="DateTime", columns="AssetName", values="Close")
assets = pivot_df.columns.tolist()
pairs = list(itertools.combinations(assets, 2))

In [31]:
pivot_df.head()

AssetName,1000SATSUSDT,1INCHUSDT,AAVEUSDT,ACAUSDT,ACEUSDT,ACHUSDT,ACMUSDT,ADAUSDT,ADXUSDT,AEURUSDT,...,XRPUSDT,XTZUSDT,XVGUSDT,XVSUSDT,YFIUSDT,YGGUSDT,ZECUSDT,ZENUSDT,ZILUSDT,ZRXUSDT
DateTime,,,,,,,,,,,,,,,,,,,,,
2017-08-18 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-19 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-20 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-21 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-22 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Testing purposes
1. Identify the pairs whose returns are close enough using euclidean distance <br>
1.a Using non-parametric SSD based <br>
1.a.1 select top 20 pairs with minimum SSD <br>
1.a.2 select pairs with frequent zero crossing **TODO** <br>
1.a.3 select pairs with high std **TODO** <br>
1.b Using Cointegration test to identify **TODO** <br>



In [32]:
distance_dict = {}
std_of_distance_dict = {}
availability_status_dict = {}

In [33]:
results = []
for a1, a2 in pairs:
    pair_df = pivot_df[[a1, a2]].dropna() # drop if pairs are empty
    if len(pair_df) < 30 * 2: # if less than 2 months of data, skip
        continue
    min_time = pair_df.index.min()
    max_time = pair_df.index.max()
    availability_status_dict[f"{a1}_{a2}"] = {"min_time": min_time, "max_time": max_time, "data_point": len(pair_df)}
    spread = pair_df[a1] - pair_df[a2]
    ssd = np.sum(spread ** 2)
    distance_dict[f"{a1}_{a2}"] = ssd
    std_of_distance_dict[f"{a1}_{a2}"] = np.std(spread)
print(f"processed total of {len(availability_status_dict)} pairs")

processed total of 39060 pairs


In [34]:
# filter for top 20 pairs with lowest SSD
sorted_distance = sorted(distance_dict.items(), key=lambda x: x[1])
top_n = 20
top_pairs = sorted_distance[:top_n]

In [35]:
print(top_pairs)

[('BTTCUSDT_PEPEUSDT', np.float64(1.8837666890294555e-20)), ('FLOKIUSDT_XECUSDT', np.float64(1.3455194522133497e-18)), ('PEPEUSDT_SHIBUSDT', np.float64(1.685197170083384e-18)), ('FLOKIUSDT_SHIBUSDT', np.float64(1.2507059655696923e-17)), ('BTTCUSDT_SHIBUSDT', np.float64(1.5306173341430674e-17)), ('FLOKIUSDT_PEPEUSDT', np.float64(2.2861218763696877e-17)), ('PEPEUSDT_XECUSDT', np.float64(2.3818872995706478e-17)), ('BTTCUSDT_FLOKIUSDT', np.float64(2.4059158129338797e-17)), ('FLOKIUSDT_WINUSDT', np.float64(6.551406222899018e-17)), ('FLOKIUSDT_LUNCUSDT', np.float64(1.5656076807723944e-16)), ('PEPEUSDT_WINUSDT', np.float64(1.6232772253715573e-16)), ('BTTCUSDT_XECUSDT', np.float64(1.9497200078939736e-16)), ('LUNCUSDT_PEPEUSDT', np.float64(2.899592835882203e-16)), ('LUNCUSDT_WINUSDT', np.float64(4.219325248236014e-16)), ('SHIBUSDT_XECUSDT', np.float64(5.814405582790432e-16)), ('LUNCUSDT_XECUSDT', np.float64(1.0428471818495377e-15)), ('LUNCUSDT_SHIBUSDT', np.float64(1.3944940894778216e-15)), ('B

In [ ]:
# define the trading rules
# for each of the pair, if the spread > 2 std, short the spread
# if the spread < -2 std, long the spread
val_pivot = validation_df.pivot(index="DateTime", columns="AssetName", values="Close")

validation_results = []
for pair_key, _ in top_pairs:
    if pair_key not in std_of_distance_dict:
        continue
    a1, a2 = pair_key.split("_")
    if a1 not in val_pivot.columns or a2 not in val_pivot.columns:
        continue

    pair_df = val_pivot[[a1, a2]].dropna().sort_index()
    if len(pair_df) < 10:
        continue

    s1 = pair_df[a1]
    s2 = pair_df[a2]
    spread = s1 - s2
    thr = 2 * std_of_distance_dict[pair_key]

    # scan and build discrete positions: +1 = long spread (long a1, short a2),
    # -1 = short spread (short a1, long a2), 
    # 0 = flat
    pos = np.zeros(len(spread), dtype=int)
    cur_pos = 0
    for i in range(len(spread)):
        cur_spread = spread.iat[i]
        if cur_pos == 0:
            if cur_spread > thr:
                cur_pos = -1  # short spread
            elif cur_spread < -thr:
                cur_pos = 1   # long spread
        else:
            # exit when spread crosses zero (sign change or exactly zero)
            prev_spread = spread.iat[i - 1] if i > 0 else cur_spread
            if cur_spread == 0 or prev_spread * cur_spread < 0:
                cur_pos = 0
        pos[i] = cur_pos

    pos_series = pd.Series(pos, index=spread.index)
    spread_ret = (s1.pct_change() - s2.pct_change()).fillna(0)
    strat_ret = pos_series.shift(1).fillna(0) * spread_ret  # apply yesterday's position to today's return

    cum_ret = (1 + strat_ret).cumprod() - 1
    final_cum = cum_ret.iloc[-1] if len(cum_ret) > 0 else 0
    sharpe = sharpe_ratio(strat_ret)

    validation_results.append({
        "pair": pair_key,
        "cum_return": final_cum,
        "sharpe": sharpe,
        "returns": strat_ret
    })

# show top validation results
validation_top = sorted(validation_results, key=lambda x: x["cum_return"], reverse=True)[:10]
for r in validation_top:
    print(f"Pair: {r['pair']}, Cumulative Return: {r['cum_return']:.2%}, Sharpe: {r['sharpe']:.2f}")
# exit when spread returns to mean


Pair: LUNCUSDT_PEPEUSDT, Cumulative Return: 1173.61%, Sharpe: 1.69
Pair: PEPEUSDT_XECUSDT, Cumulative Return: 653.35%, Sharpe: 1.47
Pair: PEPEUSDT_WINUSDT, Cumulative Return: 397.46%, Sharpe: 1.36
Pair: PEPEUSDT_SHIBUSDT, Cumulative Return: 327.35%, Sharpe: 1.23
Pair: FLOKIUSDT_LUNCUSDT, Cumulative Return: 168.85%, Sharpe: 0.98
Pair: LUNCUSDT_SHIBUSDT, Cumulative Return: 105.50%, Sharpe: 0.83
Pair: FLOKIUSDT_PEPEUSDT, Cumulative Return: 90.67%, Sharpe: 0.83
Pair: LUNCUSDT_XECUSDT, Cumulative Return: 5.50%, Sharpe: 0.30
Pair: FLOKIUSDT_WINUSDT, Cumulative Return: 4.56%, Sharpe: 0.56
Pair: LUNCUSDT_WINUSDT, Cumulative Return: 0.00%, Sharpe: -2802165968455726.00


# using cointegration approach

In [46]:
pivot_df.head()

AssetName,ADAUSDT,ALGOUSDT,APEUSDT,ATOMUSDT,AVAXUSDT,BCHUSDT,DAIUSDT,DOTUSDT,EOSUSDT,ETHUSDT,...,USDTEUR,USDTGBP,USDTJPY,USDTUSD,USTUSDT,XBTUSDT,XDGUSDT,XMRUSDT,XRPUSDT,XTZUSDT
Date,,,,,,,,,,,,,,,,,,,,,
2017-03-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.9991,NaN,NaN,NaN,NaN,NaN,NaN
2017-03-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.9990,NaN,NaN,NaN,NaN,NaN,NaN
2017-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0009,NaN,NaN,NaN,NaN,NaN,NaN
2017-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.9983,NaN,NaN,NaN,NaN,NaN,NaN
2017-04-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0008,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint

results = []
for a1, a2 in pairs:
    pair_df = pivot_df[[a1, a2]].dropna()
    if len(pair_df) < 200:
        continue

    pair_df = pair_df.sort_index()
    split_idx = int(len(pair_df) * 0.7)
    train_df = pair_df.iloc[:split_idx]
    test_df = pair_df.iloc[split_idx:]

    # Cointegration test on train/formulation period
    score, pvalue, _ = coint(train_df[a1], train_df[a2])
    if pvalue > 0.05:
        continue  # Skip non-cointegrated pairs

    # Calculate hedge ratio (beta) using OLS on train
    X = sm.add_constant(train_df[a2])
    model = sm.OLS(train_df[a1], X).fit()
    beta = model.params[a2]

    # Calculate spread and z-score on test period
    s1 = test_df[a1]
    s2 = test_df[a2]
    spread = s1 - beta * s2
    z = zscore(spread)

    # Trading logic: enter when z > 1 or z < -1, exit when z crosses 0
    position = np.where(z > 1, -1, np.where(z < -1, 1, 0))
    spread_ret = (s1.pct_change() - beta * s2.pct_change()).fillna(0)
    strat_ret = position[:-1] * spread_ret[1:]  # lag position by 1 day

    cum_ret = np.cumprod(1 + strat_ret) - 1
    sharpe = sharpe_ratio(strat_ret)

    results.append({
        "pair": (a1, a2),
        "cum_return": cum_ret.iloc[len(cum_ret)-1] if len(cum_ret) > 0 else 0,
        "sharpe": sharpe,
        "returns": strat_ret
    })

# Show top cointegrated pairs by cumulative return
top_pairs = sorted(results, key=lambda x: x["cum_return"], reverse=True)[:5]
for res in top_pairs:
    print(f"Pair: {res['pair']}, Cumulative Return: {res['cum_return']:.2%}, Sharpe Ratio: {res['sharpe']:.2f}")


/Users/ysl/opt/anaconda3/envs/py312/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:57: RuntimeWarning: overflow encountered in accumulate
  return bound(*args, **kwds)
/Users/ysl/opt/anaconda3/envs/py312/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:57: RuntimeWarning: overflow encountered in accumulate
  return bound(*args, **kwds)
/Users/ysl/opt/anaconda3/envs/py312/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:57: RuntimeWarning: overflow encountered in accumulate
  return bound(*args, **kwds)


Pair: ('ETHUSDT', 'XRPUSDT'), Cumulative Return: inf%, Sharpe Ratio: -0.73
Pair: ('LTCUSDT', 'SHIBUSDT'), Cumulative Return: inf%, Sharpe Ratio: -0.37
Pair: ('BCHUSDT', 'USDTCAD'), Cumulative Return: 3020779106659399665959339967607699945889185544848667076189289760133421903926938030769506018380480607203852209627847702334773754365595832223157537466457507857118370879096092662148101121981469184690294367799695244352947224576.00%, Sharpe Ratio: -0.18
Pair: ('BCHUSDT', 'USDTGBP'), Cumulative Return: 278476002377861905489047061354390724650499487012434797281753696431559900036154870574957533751558746913422435818105824082757583447069292669547940683767756865770174160258888613462970044548751958925861515060838400.00%, Sharpe Ratio: -0.25
Pair: ('LINKUSDT', 'XDGUSDT'), Cumulative Return: 229859406223025733431045292185337067506859482627065134417459102507854703741933428146176.00%, Sharpe Ratio: -1.42
